# Lesson 1: Binary Search and Complexity Analysis

**Source Reference:** [Data Structures and Algorithms in Python (freeCodeCamp / Jovian)](https://www.youtube.com/watch?v=pkYVOmU3MgA&list=PLWKjhJtqVAbnqBxcdjVGgT3uVR10bzTEB&index=19)

## The 6-Step Problem-Solving Method
A systematic approach to solving algorithmic problems:
1. **State the problem clearly:** Identify the exact input and output formats.
2. **Brainstorm test cases:** Cover standard examples and anticipate all edge cases (e.g., empty arrays, duplicates, bounds).
3. **Define a brute-force solution:** Explain the simplest working approach in plain English.
4. **Implement the brute-force solution:** Write the code and test it against your inputs.
5. **Analyze complexity:** Determine the Time and Space bottlenecks.
6. **Optimize:** Apply the correct data structure or algorithmic technique to reduce the complexity.

## Problem 1: Locate Card (Searching a Sorted Array)
> **Question:** Alice has some cards with numbers written on them. She arranges the cards in decreasing order and lays them out face down in a sequence on a table. She challenges Bob to pick out the card containing a given number by turning over as few cards as possible. Write a function to help Bob locate the card.

### Input and Output Formats
* **Input**:
  * `cards`: A list of numbers sorted in decreasing order (e.g., `[13, 11, 10, 7, 4, 3, 1, 0]`)
  * `query`: The target number to find (e.g., `7`)
* **Output**:
  * `position`: The 0-indexed position of `query` inside `cards`. Return `-1` if the query is not found.

In [6]:
# We represent test cases as dictionaries to automate our testing process.
tests = []

# 1. Standard Case: Query occurs in the middle
tests.append({'input': {'cards': [13, 11, 10, 7, 4, 3, 1, 0], 'query': 7}, 'output': 3})

# 2. Standard Case: Query occurs near the end
tests.append({'input': {'cards': [13, 11, 10, 7, 4, 3, 1, 0], 'query': 1}, 'output': 6})

# 3. Edge Case: Query is the first element
tests.append({'input': {'cards': [4, 2, 1, -1], 'query': 4}, 'output': 0})

# 4. Edge Case: Query is the last element
tests.append({'input': {'cards': [3, -1, -9, -127], 'query': -127}, 'output': 3})

# 5. Edge Case: Array contains exactly one element
tests.append({'input': {'cards': [6], 'query': 6}, 'output': 0})

# 6. Edge Case: Query does not exist in the list
tests.append({'input': {'cards': [9, 7, 5, 2, -9], 'query': 4}, 'output': -1})

# 7. Edge Case: The list is completely empty
tests.append({'input': {'cards': [], 'query': 7}, 'output': -1})

# 8. Edge Case: List contains repeating numbers, but the query is unique
tests.append({'input': {'cards': [8, 8, 6, 6, 6, 6, 6, 3, 2, 2, 2, 0, 0, 0], 'query': 3}, 'output': 7})

# 9. Edge Case: The query repeats multiple times. 
# RULE: Return the FIRST occurrence.
tests.append({'input': {'cards': [8, 8, 6, 6, 6, 6, 6, 6, 3, 2, 2, 2, 0, 0, 0], 'query': 6}, 'output': 2})

## Approach 1: Linear Search (Brute Force)
**Strategy**: Iterate over the list sequentially from index 0 to the end. Check each card one by one until the target is found.

### Complexity Analysis
* **Time Complexity**: $O(N)$. In the worst-case scenario (the card is at the very end, or doesn't exist), we must access the array $N$ times.
* **Space Complexity**: $O(1)$. We only require a single variable (`position`) to track our place in memory.

In [7]:
def locate_card_linear(cards, query):
    position = 0
    
    # We use a while loop with a bounds check (position < len(cards)) 
    # to prevent IndexError exceptions if the array is empty.
    while position < len(cards):
        # If the current card matches the query, return the index
        if cards[position] == query:
            return position
        
        # Move to the next card
        position += 1
        
    # If the loop finishes without returning, the target is not in the array
    return -1

## Approach 2: Binary Search (Optimized)
We can optimize this by leveraging the fact that the array is **sorted**. By checking the middle element, we can immediately eliminate half of the remaining array.

### Duplicate Handling (The "First Occurrence" Edge Case)
If the middle element equals our query, we cannot just return it immediately because duplicates might exist to its left. We must check `mid - 1`. If the left element is *also* the query, the true "first occurrence" is further left, so we discard the right half.

### Calculating Time Complexity for Binary Search
To calculate time complexity, count how many times the array size is halved until only 1 element remains.
* Initial length: $N$
* Iteration 1: $N / 2$
* Iteration 2: $N / 4$ (which is $N / 2^2$)
* Iteration 3: $N / 8$ (which is $N / 2^3$)
* Iteration $k$: $N / 2^k$

Since the final length of the array we check is $1$, we set the equation to:
$$\frac{N}{2^k} = 1$$
$$N = 2^k$$
$$\log_2 N = k$$

* **Time Complexity**: $O(\log N)$
* **Space Complexity**: $O(1)$

In [8]:
def test_location(cards, query, mid):
    """Helper logic to handle bounds and duplicates."""
    mid_number = cards[mid]
    
    if mid_number == query:
        # If the previous element is also the query, the first occurrence is further left
        if mid - 1 >= 0 and cards[mid - 1] == query:
            return 'left'
        else:
            return 'found'
            
    # Remember: The array is sorted in DECREASING order.
    elif mid_number < query:
        # If the middle is smaller than the query, the target must be on the left
        return 'left'
    else:
        # If the middle is larger than the query, the target must be on the right
        return 'right'

def locate_card_binary(cards, query):
    # Set starting bounds
    lo, hi = 0, len(cards) - 1
    
    while lo <= hi:
        # Find the middle index
        mid = (lo + hi) // 2
        result = test_location(cards, query, mid)
        
        if result == 'found':
            return mid
        elif result == 'left':
            hi = mid - 1   # Discard the right half
        elif result == 'right':
            lo = mid + 1   # Discard the left half
            
    return -1

## Generic Binary Search
Binary search is a highly reusable pattern. We can extract the loop mechanics (`while lo <= hi`) into a generic function. By passing a custom `condition` closure to this function, we can reuse this $O(\log N)$ engine to solve different problems without rewriting the while loop.

In [9]:
def binary_search(lo, hi, condition):
    """A highly reusable, generic binary search engine."""
    while lo <= hi:
        mid = (lo + hi) // 2
        result = condition(mid)
        
        if result == 'found':
            return mid
        elif result == 'left':
            hi = mid - 1
        else:
            lo = mid + 1
    return -1

def locate_card_generic(cards, query):
    """Wrapper leveraging a functional closure to inject state context."""
    def condition(mid):
        if cards[mid] == query:
            if mid > 0 and cards[mid - 1] == query:
                return 'left'
            return 'found'
        elif cards[mid] < query:
            return 'left' # Descending array logic
        else:
            return 'right'
            
    return binary_search(0, len(cards) - 1, condition)

## Problem 2: Find First and Last Position of Element in Sorted Array (LeetCode 34)

> **Question:** Given an array of integers `nums` sorted in non-decreasing order, find the starting and ending position of a given `target` value. If `target` is not found in the array, return `[-1, -1]`. You must write an algorithm with $O(\log N)$ runtime complexity.

* **Input:** `nums` = `[5, 7, 7, 8, 8, 10]`, `target` = `8`
* **Output:** `[3, 4]`

**Strategy:**
Because the array is *ascending*, the `<` logic flips compared to Problem 1. We will use our generic `binary_search` engine to run two separate searches:
1. `first_position`: If a match is found, aggressively push the boundary **left** to find the absolute start.
2. `last_position`: If a match is found, aggressively push the boundary **right** to find the absolute end.

In [10]:
def first_position(nums, target):
    def condition(mid):
        if nums[mid] == target:
            # If the element to the left is also the target, keep pushing left
            if mid > 0 and nums[mid - 1] == target:
                return 'left'
            return 'found'
        elif nums[mid] < target:
            # Ascending array: if current is smaller, target must be on the right
            return 'right'  
        else:
            return 'left'
            
    return binary_search(0, len(nums) - 1, condition)

def last_position(nums, target):
    def condition(mid):
        if nums[mid] == target:
            # If the element to the right is also the target, keep pushing right
            if mid < len(nums) - 1 and nums[mid + 1] == target:
                return 'right'
            return 'found'
        elif nums[mid] < target:
            return 'right'
        else:
            return 'left'
            
    return binary_search(0, len(nums) - 1, condition)

def first_and_last_position(nums, target):
    """Returns absolute structural coordinate boundaries as a tuple pair."""
    return first_position(nums, target), last_position(nums, target)